# Procesos de contacto
Integrantes: 
* Ilia Barrientos
* Benjamin Muñoz

 se utilizó ia en las partes de generar videos y graficos, están marcados con:
  #######  ayuda de claude #######

# PC Z2

In [ ]:
import numpy as np

In [ ]:
def realizacion(lambda_,M0,T):
    M0 = M0.copy()
    trayectorias = [M0.copy()]
    #sumamos número de infectados At
    A_t = np.sum(M0)
    tiempos = [0]
    t = 0
    while A_t > 0:
        # tasa total eventos
        R = A_t*(1+4*lambda_)
        # generamos un tiempo delta_t uniforme
        U1 = np.random.uniform(0,1)
        delta_t = -np.log(U1)/R
        if t + delta_t > T:
            break
        t = t + delta_t
        # escogemos un infectado de At al azar
        x_infectado = tuple(np.argwhere(M0 == 1)[np.random.randint(A_t)])
        # genermaos otra uniforme 
        U2 = np.random.uniform(0,1)
        if U2 < 1/(1+4*lambda_):
            #acá x se recupera, entonces At baja en 1
            A_t = A_t - 1
            M0[x_infectado] = 0 
        else:
            # acá x intenta contagiar a uno de sus 4 vecinos
            # sacamos sus 4 vecinos
            i , j = x_infectado
            N_x = [(i + 1, j) , (i-1, j) , (i,j+1) , (i,j-1)] 
            # escogemos algun vecino al azar
            vecino = N_x[np.random.randint(len(N_x))]
            a , b = vecino
            if 0 <= a < M0.shape[0] and 0 <= b < M0.shape[1]:
                # si está sano, se infecta
                if M0[vecino] == 0:
                    A_t = A_t + 1
                    M0[vecino] = 1
        trayectorias.append(M0.copy())
        tiempos.append(t)
    return trayectorias, tiempos

In [ ]:
M0 = np.random.randint(0, 2, size=(10, 10))
lambda_ = 0.5
T = 20
trayectorias, tiempos = realizacion(lambda_, M0, T)
for i in range(len(trayectorias)):
    print(f"Matriz {i}:")
    print(trayectorias[i])
#la matriz tiene 50% prob de tener casillas 1, 50% prob de tener casillas 0

### Estimación de lambda_c 

Partimos con un solo infectado en el centro y simulamos con `realizacion` hasta un tiempo `T`. Repetimos muchas veces y llamamos `P(t)` a la fracción de realizaciones en que el contagio sigue vivo al tiempo `t`. Lo hacemos para varios valores de lambda dentro de una ventana, que se puede cambiar.

Para el proceso de contacto en Z² se sabe que:
- Si lambda < lambda_c, el contagio se extingue rápido: `P(t)` decae exponencialmente .
- Si lambda > lambda_c, hay probabilidad positiva de sobrevivir para siempre, así que `P(t)` se estabiliza en un valor positivo.
- Si lambda = lambda_c, el contagio también se extingue , pero muy lentamente.

**La idea para encontrar lambda_c.** Miramos la razón

> r(t) = P(2t) / P(t)

que es la probabilidad de que el contagio siga vivo en `2t` sabiendo que seguía vivo en `t`. Cómo se comporta al aumentar `t`:
- Si lambda < lambda_c, `r(t)` **baja**: mientras más tiempo lleva vivo, más rápido se apaga.
- Si lambda > lambda_c, `r(t)` **sube hacia 1**: si sobrevivió un buen rato, casi seguro sobrevive para siempre.
- Si lambda = lambda_c, `r(t)` **ni sube ni baja**: no depende de `t` 
Entonces, si dibujamos `r(t)` contra lambda para distintos `t`, las curvas **se cruzan en lambda_c**. Ese punto de cruce es nuestra estimación.

`realizacion` guarda una copia de la matriz en cada evento y recorre toda la matriz para escoger un infectado, así que usamos una matriz y un `T` chicos (81×81, `T = 60`). Con esto el contagio no alcanza el borde cerca de lambda_c, pero la estimación es menos precisa que con un `T` más grande. Por el mismo motivo, ventanas de lambda muy por encima de lambda_c se vuelven muy lentas de simular.

In [ ]:
import matplotlib.pyplot as plt

# parámetros
T = 60                                # tiempo máximo de simulación
L = 81                                # matriz L x L, el contagio parte en el centro
N = 2000                              # realizaciones por cada lambda
lambdas = np.linspace(0.40, 0.43, 7)  # ventana de lambda a probar 

# condición inicial: 1 infectado en el centro
M_ini = np.zeros((L, L), dtype=np.int8)
M_ini[L//2, L//2] = 1

np.random.seed(1)
taus_lambda = {}      # tiempos de extinción de cada lambda (inf = sigue vivo en T)
for lambda_ in lambdas:
    taus = np.empty(N)
    for r in range(N):
        trayectorias, tiempos = realizacion(lambda_, M_ini, T)
        # si al final no queda ningún infectado, el último tiempo es el de extinción; si no, sobrevivió hasta T
        if trayectorias[-1].sum() == 0:
            taus[r] = tiempos[-1]
        else:
            taus[r] = np.inf
    taus_lambda[lambda_] = taus
    print(f"lambda = {lambda_:.4f}   fracción viva en T = {(taus > T).mean():.4f}")

### Primer criterio: razones a tiempos dobles

Con los tiempos de extinción ya tenemos `P(t)` para cualquier `t` sin volver a simular. Calculamos `r = P(2t)/P(t)` para t = 7.5, 15 y 30. Es una probabilidad condicional: de los que siguen vivos en `t`, qué fracción sigue viva en `2t`. Por eso su error estándar es el de una proporción, √(r(1−r)/n), con n = número de vivos en `t`.

Cómo leer la tabla (cada fila es un lambda): si las razones bajan de izquierda a derecha, estamos bajo lambda_c; si suben, estamos sobre lambda_c; si quedan más o menos planas, estamos cerca.

In [ ]:
t_dobles = [7.5, 15, 30]    # comparamos P(15)/P(7.5), P(30)/P(15) y P(60)/P(30)

def razon(taus, t):# los que siguen vivos en t, fracción que sigue viva en 2t
    n1 = (taus > t).sum()
    n2 = (taus > 2*t).sum()
    return n2/n1 if n1 > 0 else np.nan

razones = {}      # razones[lambda_] = lista de (razón, error estándar), una por cada t
print("  lambda   " + "".join(f"  P({2*t:g})/P({t:g})      " for t in t_dobles))
for lambda_ in lambdas:
    taus = taus_lambda[lambda_]
    fila = []
    for t in t_dobles:
        r = razon(taus, t)
        n1 = (taus > t).sum()
        fila.append((r, np.sqrt(r*(1-r)/n1)))
    razones[lambda_] = fila
    print(f"{lambda_:.4f}   " + "   ".join(f"{r:.3f} ± {e:.3f}" for r, e in fila))

### el cruce de las razones

Para cada lambda miramos cómo cambia la razón al pasar de un tiempo chico `t_a` a uno más grande `t_b`:

> d(lambda) = r(t_b) − r(t_a)

- `d < 0` si lambda < lambda_c (la razón baja con el tiempo),
- `d > 0` si lambda > lambda_c (la razón sube con el tiempo),
- `d = 0` en lambda_c, que es justo donde se cruzan las curvas de `r(t_a)` y `r(t_b)`.

Como cada `d` tiene ruido, en vez de buscar entre qué dos puntos cambia el signo ajustamos una **recta** `d ≈ a + b·lambda` por mínimos cuadrados y tomamos el lambda donde la recta vale 0: lambda_c ≈ −a/b.

**Barra de error con bootstrap.** Remuestreamos con reemplazo las `N` realizaciones de cada lambda (sin simular de nuevo), repetimos el cálculo `B` veces y miramos cuánto varía el lambda_c obtenido. Los percentiles 2.5% y 97.5% dan un intervalo del 95%.

Como chequeo de estabilidad calculamos el cruce con tres pares de tiempos: si dan parecido, la estimación es estable; si se van corriendo, hay sesgo por tiempos cortos y conviene subir `T`.

In [ ]:
# d(lambda) = r(t_b) - r(t_a)
def diferencia(taus_dict, t_a, t_b):
    return np.array([razon(taus_dict[lambda_], t_b) - razon(taus_dict[lambda_], t_a)
                     for lambda_ in lambdas])

# lambda donde la recta ajustada a d(lambda) vale 0
def cruce(taus_dict, t_a, t_b):
    d = diferencia(taus_dict, t_a, t_b)
    ok = np.isfinite(d)
    if ok.sum() < 2:
        return np.nan
    b, a = np.polyfit(lambdas[ok], d[ok], 1)    # recta d = a + b*lambda
    if b <= 0:                                  # d debería crecer con lambda; si no, no hay cruce razonable
        return np.nan
    return -a/b

# estabilidad: el cruce con distintos pares de tiempos
for t_a, t_b in [(7.5, 15), (15, 30), (7.5, 30)]:
    print(f"lambda_c con razones a t={t_a:g} vs t={t_b:g}: {cruce(taus_lambda, t_a, t_b):.4f}")

# estimación principal: t = 7.5 vs t = 30 (los más separados)
t_a, t_b = 7.5, 30
lambda_c = cruce(taus_lambda, t_a, t_b)






# bootstrap: remuestreamos las N realizaciones de cada lambda
rng = np.random.default_rng(0)
B = 500
cruces = np.empty(B)
for b in range(B):
    boot = {lambda_: rng.choice(taus_lambda[lambda_], size=N, replace=True) for lambda_ in lambdas}
    cruces[b] = cruce(boot, t_a, t_b)

validos = ~np.isnan(cruces)
print(f"\nlambda_c ≈ {lambda_c:.4f}   ({validos.sum()}/{B} remuestras con cruce válido)")
if validos.sum() > 1:
    print(f"error estándar bootstrap = {np.nanstd(cruces):.4f}")
    print(f"intervalo 95% = [{np.nanpercentile(cruces, 2.5):.4f}, {np.nanpercentile(cruces, 97.5):.4f}]")
print("valor conocido ≈ 0.412 (solo para comparar al final)")

In [ ]:
ts = np.linspace(1, T, 60)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# 1) P(t) para cada lambda
for lambda_ in lambdas:
    P = (taus_lambda[lambda_][:, None] > ts[None, :]).mean(axis=0)
    ax[0].plot(ts, P, label=f"lambda={lambda_:.3f}")
ax[0].set_xlabel("t")
ax[0].set_ylabel("P(sigue vivo al tiempo t)")
ax[0].set_title("un infectado inicial")
ax[0].legend(fontsize=8)

# 2) razones P(2t)/P(t) en función de lambda
for idx, t in enumerate(t_dobles):
    r = [razones[lambda_][idx][0] for lambda_ in lambdas]
    e = [razones[lambda_][idx][1] for lambda_ in lambdas]
    ax[1].errorbar(lambdas, r, yerr=e, marker="o", capsize=3, label=f"P({2*t:g})/P({t:g})")
ax[1].set_xlabel("lambda")
ax[1].set_ylabel("razón a tiempos dobles")
ax[1].set_title("las curvas se cruzan en lambda_c")
ax[1].legend(fontsize=8)

# 3) d(lambda) y su recta ajustada
d = diferencia(taus_lambda, t_a, t_b)
ax[2].plot(lambdas, d, "o", label=f"d = r({t_b:g}) - r({t_a:g})")
ok = np.isfinite(d)
if ok.sum() >= 2:
    pend, inter = np.polyfit(lambdas[ok], d[ok], 1)
    ax[2].plot(lambdas, inter + pend*lambdas, "-", color="C1", label="recta ajustada")
ax[2].axhline(0, color="gray", lw=1)
if not np.isnan(lambda_c):
    ax[2].axvline(lambda_c, color="red", ls="--", label=f"lambda_c ≈ {lambda_c:.3f}")
ax[2].axvline(0.412, color="green", ls=":", label="valor conocido ≈ 0.412")
ax[2].set_xlabel("lambda")
ax[2].set_ylabel("d")
ax[2].set_title("cruce por cero")
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Simulación con video de la enfermedad

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from IPython.display import HTML

In [ ]:
lambda_ = 0.5
L = 61
T_sim = 25
M_ini = np.zeros((L, L), dtype=np.int8)
M_ini[L//2, L//2] = 1

# con un solo infectado muchas realizaciones se extinguen en pocos eventos,
# así que repetimos hasta obtener una que sobreviva
np.random.seed(0)
while True:
    trayectorias, tiempos = realizacion(lambda_, M_ini, T_sim)
    if trayectorias[-1].sum() > 0:
        break


n_frames = 150
t_frames = np.linspace(0, T_sim, n_frames)
idx = np.searchsorted(tiempos, t_frames, side="right") - 1

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(trayectorias[0], cmap=ListedColormap(["#186e3f", "#5f1549"]),
               vmin=0, vmax=1, interpolation="nearest")
ax.set_xticks([]); ax.set_yticks([])
titulo = ax.set_title("")

def actualizar(k):
    M = trayectorias[idx[k]]
    im.set_data(M)
    titulo.set_text(f"t = {t_frames[k]:.1f}   infectados = {M.sum()}")
    return im, titulo

anim = FuncAnimation(fig, actualizar, frames=n_frames, interval=50)
plt.close(fig)                  # evita que se dibuje un cuadro estático extra
HTML(anim.to_jshtml())          # lo muestra dentro del notebook, con botones de play



# guardar
# anim.save("contagio.gif", writer="pillow", fps=20)
# anim.save("contagio.mp4", fps=20)    # mp4 necesita ffmpeg instalado

# Grafos aleatorios con pesos aleatorios.

## Creación del grafo, pesos y simulación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
def crear_grafo(n, c):
    p = c/n # n vértices y grado medio c
    conexion = np.triu(np.random.rand(n, n) < p, k=1)    # aristas, solo sobre la diagonal (True/False: hay o no arista)
    A = conexion | conexion.T                            # simétrica
    vecinos = [np.flatnonzero(A[x]) for x in range(n)]   # vecinos de cada vértice
    return vecinos, conexion

def asignar_tipos(vecinos, conexion, K, fracs=None):
    n = len(vecinos)
    if fracs is None:
        fracs = np.arange(1, K + 1) / (K*(K + 1)/2)
    tipos = np.random.choice(np.arange(1, K + 1), size=(n, n), p=fracs)   # tipo de cada arista posible
    A = np.where(conexion, tipos, 0)
    A = A + A.T                                                 # simétrica: A[x,v] = A[v,x] = tipo
    pesos_v = [1.0/A[x, vecinos[x]] for x in range(n)]          # 1, 1/2, ..., 1/K según el tipo de la arista
    s = np.array([w.sum() for w in pesos_v])                    # suma de pesos de los vecinos de cada vértice
    return pesos_v, s

### Ley de los tipos de contacto
Probabilidad de que una arista sea de tipo k, con k = 1..K (el tipo k tiene tasa λ/k). Esto lo decidimos nosotros y cambia las conclusiones:
- `"prop_k"`: proporcional a k (es lo que hace `asignar_tipos` por defecto). El peso medio de una arista queda 2/(K+1), así que λ_c crece con K.
- `"uniforme"`: todos los tipos igual de probables.
- `"decae"`: proporcional a 1/k², casi todo queda en los tipos bajos.

In [ ]:
def ley_rangos(K, ley="prop_k"):  # Nos confundimos con la definición de rangos con respecto al paper que teníamos, sería más bien ley de tipos, pero hicimos todo el 
    k = np.arange(1, K + 1)         # codigo usando ley de rangos, y cambiarlo es mucho  D:
    if ley == "prop_k":
        w = k.astype(float)
    elif ley == "uniforme":
        w = np.ones(K)
    elif ley == "decae":
        w = 1.0 / k**2
    else:
        raise ValueError("ley debe ser 'prop_k', 'uniforme' o 'decae'")
    return w / w.sum()

In [ ]:
def componente_gigante(vecinos):  #Vértices de la componente conexa más grande (BFS).
    n = len(vecinos)
    visto = np.zeros(n, dtype=bool)
    mejor = []
    for r in range(n):
        if visto[r]:
            continue
        comp = [r]
        visto[r] = True
        i = 0
        while i < len(comp):
            for v in vecinos[comp[i]]:
                if not visto[v]:
                    visto[v] = True
                    comp.append(v)
            i += 1
        if len(comp) > len(mejor):
            mejor = comp
    return np.array(mejor)

def vertice_tipico(vecinos): #estado0 con un único infectado: vértice de la componente gigante con grado más cercano al grado medio#
    comp = componente_gigante(vecinos)
    grados = np.array([len(v) for v in vecinos])
    x = comp[np.argmin(np.abs(grados[comp] - grados.mean()))]
    estado0 = np.zeros(len(vecinos), dtype=np.int8)
    estado0[x] = 1
    return estado0

def matriz_pesos(vecinos, pesos_v):  # matriz n x n con el peso de cada arista (0 si no hay arista)
    n = len(vecinos)
    W = np.zeros((n, n))
    for x in range(n):
        W[x, vecinos[x]] = pesos_v[x]
    return W

def cota_espectral(W): #1 / Λ_max: por debajo de este lambda el contagio se extingue rápido (cota inferior para lambda_c).
    return 1.0 / np.linalg.eigvalsh(W).max()

### La simulación: `realizacion2`

In [ ]:
def realizacion2(lambda_, estado0, vecinos, pesos_v, s, T, guardar=False):
    estado = estado0.copy()                     # vector de largo n con 0/1
    A_t = int(estado.sum())                     # número de infectados
    t = 0.0
    if guardar:
        trayectorias = [estado.copy()]
        tiempos = [0.0]
    else:
        trayectorias, tiempos = None, None
        
    while A_t > 0: #mientra hay infectados
        infectados = np.flatnonzero(estado)  
        sx = s[infectados] #lista de suma de los pesos de los vecinos del infectado x
        S = sx.sum() #suma de los pesos de los vecinos de los infectados

        # tasa total eventos: curaciones (1 por infectado) + intentos de contagio (lambda * pesos)
        R = A_t + lambda_*S 
        U1 = np.random.uniform(0, 1)
        delta_t = -np.log(U1)/R
        if t + delta_t > T:
            break

        t = t + delta_t
        U2 = np.random.uniform(0, 1) #uniforme para decidir si se cura o se infecta alguien.
        if U2 < A_t/R:  
            # se cura un infectado elegido al azar
            x = infectados[np.random.randint(A_t)]
            estado[x] = 0
            A_t = A_t - 1
        else:
            # contagio: x se elige proporcional a la suma de las tasas hacia sus vecinos
            x = infectados[np.random.choice(A_t, p=sx/S)]
            # y el vecino v se elige proporcional a su tasa (1, 1/2 o 1/3)
            v = np.random.choice(vecinos[x], p=pesos_v[x]/s[x])
            
            if estado[v] == 0:                  # si ya estaba infectado, no pasa nada
                estado[v] = 1
                A_t = A_t + 1
        if guardar:
            trayectorias.append(estado.copy())
            tiempos.append(t)

            
    t_ext = t if A_t == 0 else np.inf           # tiempo de extinción (inf si sobrevive hasta T)
    return t_ext, trayectorias, tiempos

## Estimador de λ_c
Para cada λ simulamos muchas realizaciones y guardamos el tiempo de extinción. Con eso calculamos las razones P(2t)/P(t), la diferencia d(λ) = r(t_b) − r(t_a), ajustamos una recta por mínimos cuadrados y buscamos su cero, que es la estimación de λ_c.
Usamos solo los λ **no saturados** (razones entre `r_min` y `r_max`) y, en `cruce_pico`, solo la parte donde d(λ) crece: lejos de λ_c las razones se pegan a 0 o a 1, d(λ) vuelve a bajar y eso mueve el cruce.
El intervalo de confianza sale de un bootstrap sobre los tiempos de extinción.

In [ ]:
def razon(taus, t):   # de los que siguen vivos en t, fracción que sigue viva en 2t
    n1 = (taus > t).sum()
    n2 = (taus > 2*t).sum()
    return n2/n1 if n1 > 0 else np.nan

def diferencia(taus_dict, lambdas, t_a, t_b):
    return np.array([razon(taus_dict[l], t_b) - razon(taus_dict[l], t_a) for l in lambdas])

def cruce(taus_dict, lambdas, t_a, t_b, r_min=0.02, r_max=0.98):
    lam = np.asarray(lambdas)
    ra = np.array([razon(taus_dict[l], t_a) for l in lam])
    rb = np.array([razon(taus_dict[l], t_b) for l in lam])
    d = rb - ra
    ok = np.isfinite(d) & (ra > r_min) & (ra < r_max) & (rb > r_min) & (rb < r_max)
    if ok.sum() < 2:
        return np.nan
    b, a = np.polyfit(lam[ok], d[ok], 1)       # recta d = a + b*lambda
    if b <= 0:                                  # d debería crecer con lambda
        return np.nan
    return -a/b

In [ ]:
#######  ayuda de claude #######
#  La función cruce_pico y la organización del procedimiento en estimar_lambda_c_pesos
#  se desarrollaron con ayuda de Claude. La definición de las razones, de d(lambda), el estimador por cruce, el bootstrap y la simulación con tasas
#   y normalización (realizacion2) son trabajo propio, a partir de Z2.



def cruce_pico(taus_dict, lambdas, t_a, t_b, r_min=0.02, r_max=0.98):
    # como cruce, pero la recta se ajusta solo con la parte ascendente de d(λ), hasta su máximo
    # después del máximo d vuelve a bajar porque las razones se saturan en 1, y eso mueve el cruce
    lam = np.asarray(lambdas)
    ra = np.array([razon(taus_dict[l], t_a) for l in lam])
    rb = np.array([razon(taus_dict[l], t_b) for l in lam])
    d = rb - ra
    ok = np.isfinite(d) & (ra > r_min) & (ra < r_max) & (rb > r_min) & (rb < r_max)
    if ok.sum() < 2:
        return np.nan
    idx = np.flatnonzero(ok)
    sel = idx[idx <= idx[np.argmax(d[idx])]]
    if len(sel) < 2:
        return np.nan
    b, a = np.polyfit(lam[sel], d[sel], 1)
    return -a/b if b > 0 else np.nan

def estimar_lambda_c_pesos(vecinos, pesos_v, s, estado0, T, N, t_dobles, t_a, t_b, pares, lambdas,
                           B=500, verbose=True):
    # estima lambda_c de un grafo con pesos (pesos_v, s) usando realizacion2
    # lambdas es la ventana de lambda que elegimos a mano a partir del barrido grueso
    lambdas = np.asarray(lambdas)

    # 1) taus: tiempos de extinción (inf = sigue vivo en T)
    taus_lambda = {}
    for l in lambdas:
        taus = np.empty(N)
        for r in range(N):
            taus[r] = realizacion2(l, estado0, vecinos, pesos_v, s, T)[0]
        taus_lambda[l] = taus
        if verbose:
            print(f"  lambda = {l:.4f}   fracción viva en T = {(taus > T).mean():.4f}")

    # 2) razones con su error estándar
    razones = {}
    if verbose:
        print("  lambda   " + "".join(f"  P({2*t:g})/P({t:g})      " for t in t_dobles))
    for l in lambdas:
        fila = []
        for t in t_dobles:
            r = razon(taus_lambda[l], t)
            n1 = (taus_lambda[l] > t).sum()
            fila.append((r, np.sqrt(r*(1-r)/n1) if n1 > 0 else np.nan))
        razones[l] = fila
        if verbose:
            print(f"  {l:.4f}   " + "   ".join(f"{r:.3f} ± {e:.3f}" for r, e in fila))

    # 3) estabilidad con distintos pares de tiempos
    if verbose:
        for ta, tb in pares:
            print(f"  lambda_c con t={ta:g} vs t={tb:g}: {cruce_pico(taus_lambda, lambdas, ta, tb):.4f}")

    # 4) estimación principal (rama ascendente) + bootstrap; el cruce global se guarda para comparar
    lambda_c = cruce_pico(taus_lambda, lambdas, t_a, t_b)
    rng = np.random.default_rng(0)
    cruces = np.empty(B)
    for b in range(B):
        boot = {l: rng.choice(taus_lambda[l], size=N, replace=True) for l in lambdas}
        cruces[b] = cruce_pico(boot, lambdas, t_a, t_b)
    cruces[(cruces < lambdas[0]) | (cruces > lambdas[-1])] = np.nan       # cruce fuera de la ventana: no vale
    validos = ~np.isnan(cruces)
    if validos.sum() > 1:
        se = np.nanstd(cruces)
        ic = (np.nanpercentile(cruces, 2.5), np.nanpercentile(cruces, 97.5))
    else:
        se, ic = np.nan, (np.nan, np.nan)
    if verbose:
        print(f"\n  lambda_c ≈ {lambda_c:.4f}   ee bootstrap = {se:.4f}   "
              f"IC95% = [{ic[0]:.4f}, {ic[1]:.4f}]   ({validos.sum()}/{B} válidas)")
    if np.isnan(lambda_c) or not (lambdas[0] < lambda_c < lambdas[-1]):
        print("  el cruce cae fuera de la ventana o no existe")

    return dict(lambdas=lambdas, taus_lambda=taus_lambda, razones=razones, lambda_c=lambda_c,
                lambda_c_global=cruce(taus_lambda, lambdas, t_a, t_b), se=se, ic=ic, cruces=cruces,
                validas=int(validos.sum()), t_dobles=t_dobles, t_a=t_a, t_b=t_b, T=T)

## Grado medio: cómo varía λ_c con c

Ahora variamos el grado medio c del grafo, con **K = 1** (todas las aristas con tasa λ). Lo hacemos por pasos:

1. **Barrido grueso** de λ para cada c, mirando qué fracción de las realizaciones sigue viva en T.
2. **Ventanas** de λ elegidas a mano a partir del barrido.
3. **Cálculo de λ_c** en cada ventana, con `N_GRAFOS` grafos aleatorios distintos por cada c para ver que sea consistente.
4. **Gráfico** de λ_c vs c.

En la tabla comparamos con la aproximación de campo medio 1/(c+1) y con la cota espectral 1/Λ_max
(Λ_max es el mayor valor propio de la matriz de adyacencia; por debajo de 1/Λ_max el contagio se extingue rápido).

In [ ]:
RAPIDO = False                  # True para probar rápido (menos simulaciones, más ruido)
if RAPIDO:
    N_s, B_s, Ng_s = 100, 200, 50
    N_GRAFOS = 1                # grafos aleatorios distintos por cada grado medio
else:
    N_s, B_s, Ng_s = 500, 500, 100
    N_GRAFOS = 3

n = 500
T = 40
t_dobles = [5, 10, 20]
t_a, t_b = 5, 20
pares = [(5, 10), (10, 20), (5, 20)]
MALLA = (0.6, 3.0, 11)          # barrido grueso: de 0.6 a 3 veces la cota espectral 1/Λ_max

### Paso 1: barrido grueso
Para cada c simulamos con pocas realizaciones en una malla ancha de λ (de `MALLA[0]` a `MALLA[1]` veces la cota espectral) y miramos qué fracción sigue viva en T.
Con eso elegimos las ventanas del paso 2: desde donde casi no sobrevive nadie hasta donde sobrevive ~30 %, de modo que λ_c quede adentro.

In [ ]:
cs = [4, 6, 8, 10, 15, 20]
casos = [(c, 1) for c in cs]  

In [ ]:
cs = [4, 6, 8, 10, 15, 20]
casos = [(c, 1) for c in cs]                     # K = 1: todas las aristas con tasa λ

def crear_caso(c, K=1, g=0):  # g-ésimo grafo aleatorio de grado medio c, con K tipos de arista (ley prop_k)
    np.random.seed(100 + c + 1000*g)
    vecinos, conexion = crear_grafo(n, c)
    estado0 = vertice_tipico(vecinos)
    np.random.seed(1000 + 10*c + K + 1000*g)
    pesos_v, s = asignar_tipos(vecinos, conexion, K, ley_rangos(K, "prop_k"))
    return vecinos, pesos_v, s, estado0

barrido = {}
for (c, K) in casos:
    vecinos, pesos_v, s, estado0 = crear_caso(c, K)
    cota = cota_espectral(matriz_pesos(vecinos, pesos_v))
    lambdas_g = cota*np.linspace(*MALLA)
    frac = []
    for l in lambdas_g:
        taus = np.array([realizacion2(l, estado0, vecinos, pesos_v, s, T)[0] for _ in range(Ng_s)])
        frac.append((taus > T).mean())
    barrido[(c, K)] = (lambdas_g, np.array(frac))
    print(f"c = {c:>2}   cota espectral = {cota:.4f}")
    print("   lambda:   " + " ".join(f"{l:6.3f}" for l in lambdas_g))
    print("   viva en T:" + " ".join(f"{f:6.2f}" for f in np.array(frac)))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for (c, K), (lg, fr) in barrido.items():
    ax.plot(lg, fr, "o-", ms=3, label=f"c = {c}")
ax.axhline(0.3, color="k", ls=":", lw=1)
ax.set_xlabel("lambda"); ax.set_ylabel("fracción viva en T")
ax.set_title("Barrido grueso, K = 1"); ax.legend()
plt.tight_layout()
plt.show()

### Paso 2: ventanas de λ (a mano)
Con el barrido de arriba elegimos a mano la ventana de λ de cada c, desde donde casi no sobrevive nadie hasta donde sobrevive ~30 %.

In [ ]:
VENTANAS = {
    (4, 1):  np.linspace(0.200, 0.360, 10),
    (6, 1):  np.linspace(0.150, 0.270, 10),
    (8, 1):  np.linspace(0.115, 0.200, 10),
    (10, 1): np.linspace(0.095, 0.165, 10),
    (15, 1): np.linspace(0.050, 0.100, 10),
    (20, 1): np.linspace(0.040, 0.078, 10),
}

### Paso 3: λ_c en cada ventana
Con `estimar_lambda_c_pesos` calculamos λ_c de cada c en `N_GRAFOS` grafos aleatorios distintos, usando la misma ventana.
Los resultados quedan en `resultados_c[(c, K)]` (una lista con un elemento por grafo).

In [ ]:
import pickle

In [ ]:
import pickle
VER_DETALLE = False        # True: imprime la tabla de razones de cada caso
resultados_c = {}
for (c, K) in casos:
    resultados_c[(c, K)] = []
    for g in range(N_GRAFOS):
        vecinos, pesos_v, s, estado0 = crear_caso(c, K, g)
        c_real = np.mean([len(v) for v in vecinos])
        wmedio = np.concatenate(pesos_v).mean()
        cota = cota_espectral(matriz_pesos(vecinos, pesos_v))
        np.random.seed(5000 + 10*c + K + 1000*g)
        if VER_DETALLE:
            print(f"\n===== c = {c}, K = {K}, grafo {g} =====")
        res = estimar_lambda_c_pesos(vecinos, pesos_v, s, estado0, T, N_s, t_dobles, t_a, t_b, pares,
                                     lambdas=VENTANAS[(c, K)], B=B_s, verbose=VER_DETALLE)
        res.update(K=K, ley="prop_k", wmedio=wmedio, c_real=c_real, cota=cota)
        resultados_c[(c, K)].append(res)
        print(f"c = {c:>2} (real {c_real:.1f})  K = {K}  grafo {g}  lambda_c = {res['lambda_c']:.4f}  "
              f"IC95% = [{res['ic'][0]:.4f}, {res['ic'][1]:.4f}]  ({res['validas']}/{B_s} válidas)")

    # vamos guardando lo que va listo
    with open("resultados_c.pkl", "wb") as f:
        pickle.dump(resultados_c, f)
    #resultados_c = pickle.load(open("resultados_c.pkl", "rb"))  # para recuperarlo

In [ ]:
resultados_c = pickle.load(open("resultados_c.pkl", "rb"))

### Paso 4: tabla y gráfico
λ_c de cada c = promedio sobre los grafos (la desviación mide cuánto cambia de un grafo a otro).

In [ ]:
#######  ayuda de claude #######
def resumen(c, K=1):  # promedio y desviación (entre grafos) de lambda_c, y promedio de c real y cota espectral
    rs = resultados_c[(c, K)]
    lcs = np.array([r["lambda_c"] for r in rs])
    return dict(lc=np.nanmean(lcs), sd=np.nanstd(lcs) if len(lcs) > 1 else 0.0, lcs=lcs,
                c_real=np.mean([r["c_real"] for r in rs]),
                cota=np.mean([r["cota"] for r in rs]))

R1 = {c: resumen(c, 1) for c in cs}

print(f"{'c':>3} {'c real':>7} {'1/(c+1)':>8} {'1/Λmax':>8} {'λc':>8} {'desv.':>7} {'λc·(c+1)':>9}")
for c in cs:
    r = R1[c]
    print(f"{c:>3} {r['c_real']:>7.1f} {1/(c+1):>8.4f} {r['cota']:>8.4f} {r['lc']:>8.4f} {r['sd']:>7.4f} {r['lc']*(c+1):>9.3f}")

In [ ]:
#######  ayuda de claude #######
cr = np.array([R1[c]["c_real"] for c in cs])
lc = np.array([R1[c]["lc"] for c in cs])
sd = np.array([R1[c]["sd"] for c in cs])
fig, ax = plt.subplots(figsize=(7, 4.5))

for c in cs:                                                       # un punto por grafo
    ax.plot([R1[c]["c_real"]]*len(R1[c]["lcs"]), R1[c]["lcs"], ".", color="hotpink", alpha=0.5)
ax.errorbar(cr, lc, yerr=sd, marker="o", capsize=3, color="deeppink", label="simulación (promedio entre grafos)")

ax.set_xlabel("grado medio c (K=1)"); ax.set_ylabel("lambda_c")
ax.set_title("lambda_c vs grado medio"); ax.legend()
plt.tight_layout()
plt.show()

### Propagación con grado medio alto y tasa chiquita

Para ver lo que dice el gráfico, simulamos la epidemia con **la misma tasa λ muy pequeña** en dos grafos de 60 vértices con todas las aristas iguales (K = 1): uno con pocas conexiones (grado medio 4) y otro con muchas (grado medio 15). Con λ tan chica la epidemia no prende en el grafo con pocas conexiones, pero sí en el de muchas, porque ahí λ_c es mucho menor.

Verde = sano, morado = infectado.

para hacer los videos durante el resto


In [ ]:
def layout_resorte(vecinos, iteraciones=250, semilla=0):
    n = len(vecinos)
    rng = np.random.default_rng(semilla)
    pos = rng.uniform(-1, 1, size=(n, 2))
    A = np.zeros((n, n))
    for x in range(n):
        A[x, vecinos[x]] = 1.0
    k = 1.0/np.sqrt(n)
    temp = 0.1
    dtemp = temp/(iteraciones + 1)
    for _ in range(iteraciones):
        delta = pos[:, None, :] - pos[None, :, :]
        dist = np.linalg.norm(delta, axis=-1) + 1e-9
        fuerza = (k**2/dist**2 - A*dist/k)[:, :, None]*delta     
        desp = fuerza.sum(axis=1)
        largo = np.linalg.norm(desp, axis=1)[:, None] + 1e-9
        pos += desp/largo*np.minimum(largo, temp)
        temp -= dtemp
    pos -= pos.min(axis=0)
    pos /= pos.max(axis=0)
    return pos

def animar_epidemia(vecinos, pesos_v, trayectorias, tiempos, T, n_frames=60, semilla=0,
                    etiquetas_peso=None, archivo_gif="epidemia.gif"):
    from matplotlib.animation import FuncAnimation, PillowWriter
    from matplotlib.collections import LineCollection
    from matplotlib.lines import Line2D

    n = len(vecinos)
    pos = layout_resorte(vecinos, semilla=semilla)                
    tiempos = np.asarray(tiempos)
    t_max = min(T, tiempos[-1])
    ts = np.linspace(0, t_max, n_frames)                           
    estados = np.array([trayectorias[max(np.searchsorted(tiempos, t, side="right") - 1, 0)] for t in ts])  

    # un estilo de línea por tipo de arista: (forma, grosor, color)
    estilos = [("-", 2.4, "0.15"), ("--", 1.5, "0.30"), (":", 1.5, "0.40")]      
    pesos_distintos = sorted({round(float(w), 3) for x in range(n) for w in pesos_v[x]}, reverse=True)   
    estilo_de = {w: estilos[min(i, len(estilos) - 1)] for i, w in enumerate(pesos_distintos)}            

    fig, ax = plt.subplots(figsize=(8, 8), dpi=90)
    for w in pesos_distintos:                                    
        segs = [[pos[x], pos[v]] for x in range(n) for v, wv in zip(vecinos[x], pesos_v[x])
                if x < v and round(float(wv), 3) == w]            
        ls, lw, col = estilo_de[w]
        ax.add_collection(LineCollection(segs, colors=col, linestyles=ls, linewidths=lw, zorder=1))
    sc = ax.scatter(pos[:, 0], pos[:, 1], s=90, c="lightsteelblue", edgecolors="k", linewidths=0.5, zorder=3)   
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    if etiquetas_peso:                                             
        ax.legend(handles=[Line2D([0], [0], color=col, ls=ls, lw=lw, label=txt)
                           for (w, txt), (ls, lw, col) in zip(sorted(etiquetas_peso.items(), reverse=True), estilos)],
                  loc="upper right", fontsize=8)
    titulo = ax.set_title("")

    def actualizar(i):
        sc.set_facecolor(np.where(estados[i] == 1, "crimson", "lightsteelblue"))  
        titulo.set_text(f"t = {ts[i]:.1f}   infectados: {int(estados[i].sum())}")
        return sc, titulo

    anim = FuncAnimation(fig, actualizar, frames=n_frames, interval=80, blit=False)
    anim.save(archivo_gif, writer=PillowWriter(fps=6)) 
    plt.close(fig)
    try:
        from IPython.display import Image, display
        display(Image(filename=archivo_gif, width=700))
    except ImportError:
        pass
    return archivo_gif

In [ ]:
n_video = 60                  # vértices de cada grafo 
cs_video = [4, 15]            # grado medio de cada grafo: pocas conexiones vs muchas conexiones
T_video_c = 25                # tiempo que dura la simulación
MIN_INFECTADOS_c = 15         # contamos que la epidemia "prende" si llega a tantos infectados a la vez
MAX_INTENTOS_c = 30           # intentos; si ninguno prende, se muestra el que llegó más lejos

lambda_c_video = 1.5*R1[15]["lc"]       # tasa chiquita: un poco sobre λ_c del grafo con grado medio 15
if np.isnan(lambda_c_video):
    lambda_c_video = 0.1
print(f"λ = {lambda_c_video:.3f} en los dos grafos\n")

sim_c = {}
for c in cs_video:
    np.random.seed(300 + c)
    vecinos_c_v, conexion_c_v = crear_grafo(n_video, c)
    estado0_c_v = vertice_tipico(vecinos_c_v)
    pesos_c_v = [np.ones(len(v)) for v in vecinos_c_v]                       # K = 1: todas las aristas con tasa λ
    s_c_v = np.array([len(v) for v in vecinos_c_v], dtype=float)
    mejor = None
    for intento in range(1, MAX_INTENTOS_c + 1):
        t_ext, tray, tiempos = realizacion2(lambda_c_video, estado0_c_v, vecinos_c_v, pesos_c_v, s_c_v, T_video_c, guardar=True)
        maximo = max(e.sum() for e in tray)
        if mejor is None or maximo > mejor[0]:
            mejor = (maximo, tray, tiempos)
        if maximo >= MIN_INFECTADOS_c:
            break
    sim_c[c] = (vecinos_c_v, mejor[1], mejor[2])
    print(f"grado medio {c:>2} (real {np.mean([len(v) for v in vecinos_c_v]):.1f})   intentos: {intento}   máx. infectados = {mejor[0]}   infectados al final = {mejor[1][-1].sum()}")

In [ ]:
#######  ayuda de claude #######
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from IPython.display import Image, display

VERDE, MORADO, ROSA = "#69d3e6", "#E20404", "#7e7474"      # sano, infectado, aristas
nf_c = 60                                                  # fotogramas
ts_c = np.linspace(0, T_video_c, nf_c)

fig, axs = plt.subplots(1, len(cs_video), figsize=(6.5*len(cs_video), 7), dpi=80)
fig.subplots_adjust(top=0.84, bottom=0.1, wspace=0.05)
scs, estados_c, titulos = [], [], []
for ax, c in zip(axs, cs_video):
    vecinos_c_v, tray, tiempos = sim_c[c]
    tiempos = np.asarray(tiempos)
    estados_c.append(np.array([tray[max(np.searchsorted(tiempos, t, side="right") - 1, 0)] for t in ts_c]))
    pos = layout_resorte(vecinos_c_v, semilla=0)
    lo, hi = np.percentile(pos, 3, axis=0), np.percentile(pos, 97, axis=0)       # los vértices sueltos no deben achicar el dibujo
    pos = np.clip((pos - lo)/(hi - lo), -0.02, 1.02)
    segs = [[pos[x], pos[v]] for x in range(len(vecinos_c_v)) for v in vecinos_c_v[x] if x < v]
    ax.add_collection(LineCollection(segs, colors=ROSA, linewidths=0.8, alpha=0.5, zorder=1))
    scs.append(ax.scatter(pos[:, 0], pos[:, 1], s=90, c=VERDE, edgecolors="k", linewidths=0.5, zorder=3))
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    titulos.append(ax.set_title("", fontsize=12))
fig.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=VERDE, markeredgecolor="k", markersize=10, label="sano"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor=MORADO, markeredgecolor="k", markersize=10, label="infectado")],
           loc="lower center", ncol=2, frameon=False, fontsize=10)
fig.suptitle(f"Misma tasa λ = {lambda_c_video:.3f} en los dos grafos", fontsize=14, fontweight="bold")

def actualizar(i):
    for sc, est, tit, c in zip(scs, estados_c, titulos, cs_video):
        sc.set_facecolor(np.where(est[i] == 1, MORADO, VERDE))
        tit.set_text(f"grado medio {c}\nt = {ts_c[i]:.1f}   infectados: {int(est[i].sum())} de {n_video}")
    return scs

anim = FuncAnimation(fig, actualizar, frames=nf_c, interval=200, blit=False)
anim.save("video_grado_medio.gif", writer=PillowWriter(fps=5))
plt.close(fig)
display(Image(filename="video_grado_medio.gif", width=900))

## Distintos pesos de contagio

Hasta acá todas las aristas contagiaban con la misma tasa λ (K = 1). Ahora queremos ver cómo cambia λ_c cuando asignamos distintos pesos de contagio a los contactos.

Para nuestro caso decidimos **disminuir el peso medio** de las aristas, diferenciando las relaciones (aristas) en **3 tipos**, que para nosotros era suficiente. Por encima, los pensamos como:

- **tipo 1, familia**: tasa λ (peso 1)
- **tipo 2, amigos**: tasa λ/2 (peso 1/2)
- **tipo 3, compañeros**: tasa λ/3 (peso 1/3)

Esto no necesariamente es así en la realidad (hay amigos que se ven más que la familia, por ejemplo): es una simplificación para tener algo con qué interpretar los pesos.

Usamos **un mismo grafo** (el de grado medio 10) y siempre las mismas aristas; lo único que cambia es el tipo de cada arista, así que cualquier diferencia en λ_c viene solo de los pesos. Comparamos cinco composiciones:

- **base**: todas las aristas son familia (todo λ, el K = 1 de siempre)
- **equilibrada**: un tercio de cada tipo
- **mucha familia**, **muchos amigos**, **muchos compañeros**: 60 % del tipo que da el nombre y 20 % de cada uno de los otros

Lo que queremos ver es si se cumple

λc ponderada(K=3) · peso medio = λc base(K=1)

In [ ]:
c_com = 10                                  # mismo grafo que el de grado medio 10
composiciones = {                           # fracción de aristas de tipo 1, 2 y 3 (familia, amigos, compañeros)
    "base (todo λ)":     (1.0, 0.0, 0.0),
    "equilibrada":       (1/3, 1/3, 1/3),
    "mucha familia":     (0.6, 0.2, 0.2),
    "muchos amigos":     (0.2, 0.6, 0.2),
    "muchos compañeros": (0.2, 0.2, 0.6),
}


In [ ]:
c_com = 10                                  # mismo grafo que el de grado medio 10
composiciones = {                           # fracción de aristas de tipo 1, 2 y 3 (familia, amigos, compañeros)
    "base (todo λ)":     (1.0, 0.0, 0.0),
    "equilibrada":       (1/3, 1/3, 1/3),
    "mucha familia":     (0.6, 0.2, 0.2),
    "muchos amigos":     (0.2, 0.6, 0.2),
    "muchos compañeros": (0.2, 0.2, 0.6),
}

np.random.seed(100 + c_com)
vecinos_m, conexion_m = crear_grafo(n, c_com)
estado0_m = vertice_tipico(vecinos_m)

# mismas aristas, cambia solo el tipo de cada una
grafos_com = {}
for i, (nombre, fracs) in enumerate(composiciones.items()):
    np.random.seed(2000 + i)
    pesos_v, s = asignar_tipos(vecinos_m, conexion_m, 3, np.array(fracs))
    wmedio = np.concatenate(pesos_v).mean()
    cota = cota_espectral(matriz_pesos(vecinos_m, pesos_v))
    grafos_com[nombre] = (pesos_v, s, wmedio, cota)

# barrido grueso de cada composición
barrido_com = {}
for nombre, (pesos_v, s, wmedio, cota) in grafos_com.items():
    lambdas_g = cota*np.linspace(*MALLA)
    frac = []
    for l in lambdas_g:
        taus = np.array([realizacion2(l, estado0_m, vecinos_m, pesos_v, s, T)[0] for _ in range(Ng_s)])
        frac.append((taus > T).mean())
    barrido_com[nombre] = (lambdas_g, np.array(frac))
    print(f"{nombre:<18} peso medio = {wmedio:.3f}   cota espectral = {cota:.4f}")
    print("   lambda:   " + " ".join(f"{l:6.3f}" for l in lambdas_g))
    print("   viva en T:" + " ".join(f"{f:6.2f}" for f in np.array(frac)))

Ventanas de λ de cada composición, también elegidas a mano a partir del barrido de arriba.

In [ ]:
VENTANAS_COM = {
    "base (todo λ)":     np.linspace(0.076, 0.151, 10),
    "equilibrada":       np.linspace(0.124, 0.247, 10),
    "mucha familia":     np.linspace(0.099, 0.197, 10),
    "muchos amigos":     np.linspace(0.134, 0.266, 10),
    "muchos compañeros": np.linspace(0.152, 0.302, 10),
}

In [ ]:
res_com = {}
for i, (nombre, (pesos_v, s, wmedio, cota)) in enumerate(grafos_com.items()):
    np.random.seed(8000 + i)
    res = estimar_lambda_c_pesos(vecinos_m, pesos_v, s, estado0_m, T, N_s, t_dobles, t_a, t_b, pares,
                                 lambdas=VENTANAS_COM[nombre], B=B_s, verbose=False)
    res_com[nombre] = dict(res, wmedio=wmedio)
    print(f"{nombre:<18} lambda_c = {res['lambda_c']:.4f}   IC95% = [{res['ic'][0]:.4f}, {res['ic'][1]:.4f}]  ({res['validas']}/{B_s} válidas)")

print()
print(f"{'composición':<18} {'peso medio':>11} {'λc':>8} {'λc·w̄':>8} {'λc·w̄/λc(base)':>15}")
lc_b = res_com["base (todo λ)"]["lambda_c"]
for k, r in res_com.items():
    print(f"{k:<18} {r['wmedio']:>11.3f} {r['lambda_c']:>8.4f} {r['lambda_c']*r['wmedio']:>8.4f} {r['lambda_c']*r['wmedio']/lc_b:>15.3f}")

## Videos de la comparación

Simulamos la epidemia en el mismo grafo con las cuatro composiciones (base, mucha familia, muchos amigos y muchos compañeros), con el **mismo λ** en las cuatro, ubicado justo debajo de λ_c(muchos amigos), y con el mismo infectado inicial.
La idea es ver que una pandemia donde ves poco a tus contactos (más compañeros y menos familia) ayuda a disminuir el contagio, porque sube λ_c: con ese λ la epidemia sigue donde hay mucha familia, casi se apaga con muchos amigos y se apaga con muchos compañeros.

En los videos: verde = sano, morado = infectado. Las aristas se distinguen por tipo: línea continua gruesa = familia, rayas = amigos, puntos = compañeros.

λ_c de cada composición (más compañeros, menor peso medio y mayor λ_c) y el λ que usamos en los videos. Si λ queda por debajo de λ_c la epidemia se extingue.

In [ ]:
#######  ayuda de claude #######
lc = np.array([res_com[k]["lambda_c"] for k in nombres_video])
lo = np.array([res_com[k]["ic"][0] for k in nombres_video])
hi = np.array([res_com[k]["ic"][1] for k in nombres_video])
wm = np.array([res_com[k]["wmedio"] for k in nombres_video])

x = np.arange(len(nombres_video))
fig, ax = plt.subplots(figsize=(8, 4.8))
barras = ax.bar(x, lc, yerr=[np.maximum(lc - lo, 0), np.maximum(hi - lc, 0)], capsize=4,
                color=["#f8bbd0", "#f48fb1", "#ec407a", "#ad1457"], edgecolor="k")
ax.axhline(lambda_video, color="#7b1fa2", ls="--", label=f"λ de los videos = {lambda_video:.3f}")
for xi, v in zip(x, lc):
    ax.text(xi, v*1.02, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"{k}\npeso medio {w:.2f}" for k, w in zip(nombres_video, wm)])
ax.set_ylabel("lambda_c")
ax.set_title("λ_c según cuánto ves a tus contactos (mismo grafo)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
T_video = 30                  # tiempo que dura cada simulación
FRACCION_LAMBDA = 0.95        # λ de los videos = FRACCION_LAMBDA · λ_c(muchos amigos)
MIN_INFECTADOS = 10           # contamos que la epidemia arranca si en algún momento hay al menos 10 infectados a la vez
MAX_INTENTOS = 30             # intentos para que arranque desde el infectado inicial
nombres_video = ["base (todo λ)", "mucha familia", "muchos amigos", "muchos compañeros"]

# λ un poco debajo de λ_c(muchos amigos): amigos casi se apaga, compañeros (λ_c más alto) se apaga y familia (λ_c más bajo) sigue
lc_amigos = res_com["muchos amigos"]["lambda_c"]
if np.isnan(lc_amigos):                                  # si no hay λ_c, se usa el centro de la ventana
    lc_amigos = np.mean(VENTANAS_COM["muchos amigos"])
lambda_video = FRACCION_LAMBDA*lc_amigos
print(f"λ de los videos = {lambda_video:.4f} (el mismo en los cuatro)")
for nombre in nombres_video:
    print(f"   λ_c de {nombre:<18} = {res_com[nombre]['lambda_c']:.4f}")
print()

sim_video = {}
for i, nombre in enumerate(nombres_video):
    pesos_v, s, wmedio, cota = grafos_com[nombre]
    np.random.seed(9000 + i)
    for intento in range(1, MAX_INTENTOS + 1):
        t_ext, tray, tiempos = realizacion2(lambda_video, estado0_m, vecinos_m, pesos_v, s, T_video, guardar=True)
        if max(e.sum() for e in tray) >= MIN_INFECTADOS:
            break
    sim_video[nombre] = (tray, tiempos)
    arranco = max(e.sum() for e in tray) >= MIN_INFECTADOS
    print(f"{nombre:<18} peso medio = {wmedio:.3f}   intentos hasta que arranca: {intento if arranco else 'no arrancó'}   "
          f"máx. infectados = {max(e.sum() for e in tray)}   infectados al final = {tray[-1].sum()}")

In [ ]:
#######  ayuda de claude #######
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from IPython.display import Image, display

VERDE, MORADO, ROSA = "#5dd5eb", "#e0212a", "#7e7474"      # sano, infectado, aristas
pos_v = layout_resorte(vecinos_m, semilla=0)               # mismas posiciones en los cuatro videos
nf_v = 60                                                  # fotogramas por video
ts_v = np.linspace(0, T_video, nf_v)

# estilo de cada tipo de arista: (forma de la línea, grosor, transparencia, nombre en la leyenda)
estilo_tipo = {1.0:   ("-",  2.6, 0.95, "familia (λ)"),             # continua y gruesa
               0.5:   ("--", 1.6, 0.85, "amigos (λ/2)"),            # rayas
               0.333: (":",  1.6, 0.85, "compañeros (λ/3)")}        # puntos

def video_composicion(nombre, archivo):
    pesos_v, s, wmedio, cota = grafos_com[nombre]
    tray, tiempos = sim_video[nombre]
    tiempos = np.asarray(tiempos)
    estados = np.array([tray[max(np.searchsorted(tiempos, t, side="right") - 1, 0)] for t in ts_v])

    fig, ax = plt.subplots(figsize=(8.5, 9), dpi=90)           # más grande
    fig.subplots_adjust(top=0.88, bottom=0.12)

    tipos_presentes = []
    for w_tipo, (ls, lw, alfa, etiqueta) in estilo_tipo.items():       # una colección de líneas por tipo de arista
        segs = [[pos_v[x], pos_v[v]] for x in range(len(vecinos_m)) for v, w in zip(vecinos_m[x], pesos_v[x])
                if x < v and abs(w - w_tipo) < 0.01]                  # x < v: cada arista una sola vez
        if len(segs) > 0:                                             # en "base" solo hay familia
            ax.add_collection(LineCollection(segs, colors=ROSA, linestyles=ls, linewidths=lw, alpha=alfa, zorder=1))
            tipos_presentes.append(Line2D([0], [0], color=ROSA, ls=ls, lw=lw, label=etiqueta))

    sc = ax.scatter(pos_v[:, 0], pos_v[:, 1], s=40, c=VERDE, edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    estados_leyenda = [Line2D([0], [0], marker="o", color="w", markerfacecolor=VERDE, markeredgecolor="k", markersize=10, label="sano"),
                       Line2D([0], [0], marker="o", color="w", markerfacecolor=MORADO, markeredgecolor="k", markersize=10, label="infectado")]
    fig.legend(handles=tipos_presentes + estados_leyenda, loc="lower center", ncol=3, frameon=False, fontsize=10)
    titulo = ax.set_title("", fontsize=12)

    def actualizar(i):
        sc.set_facecolor(np.where(estados[i] == 1, MORADO, VERDE))
        titulo.set_text(f"{nombre}  (peso medio {wmedio:.2f}, λ = {lambda_video:.3f})\n"
                        f"t = {ts_v[i]:.1f}   infectados: {int(estados[i].sum())} de {len(estados[i])}")
        return sc, titulo

    anim = FuncAnimation(fig, actualizar, frames=nf_v, interval=200, blit=False)
    anim.save(archivo, writer=PillowWriter(fps=5))                    # fps más bajo = más lento
    plt.close(fig)
    display(Image(filename=archivo, width=750))

for nombre, archivo in zip(nombres_video, ["video_base.gif", "video_mucha_familia.gif",
                                           "video_muchos_amigos.gif", "video_muchos_companeros.gif"]):
    print(nombre)
    video_composicion(nombre, archivo)
for nombre, archivo in zip(nombres_video, ["video_base.gif", "video_mucha_familia.gif",
                                           "video_muchos_amigos.gif", "video_muchos_companeros.gif"]):
    print(nombre)
    video_composicion(nombre, archivo)

## Simulación estándar de nuestro modelo

Una realización en un grafo chico con los tres tipos de contacto, con las aristas dibujadas más gruesas y oscuras mientras más cercano es el contacto (continua = familia, rayas = amigos, puntos = compañeros), y los infectados en rojo.

In [ ]:
# grafo aleatorio chico con tres tipos de contacto (familia, amigos, compañeros)
np.random.seed(7)
n_v, c_v, T_v = 80, 6, 20
vecinos_v, conexion_v = crear_grafo(n_v, c_v)
pesos_v_v, s_v = asignar_tipos(vecinos_v, conexion_v, 3, ley_rangos(3, "prop_k"))
estado0_v = vertice_tipico(vecinos_v)
lambda_v = 2.0*cota_espectral(matriz_pesos(vecinos_v, pesos_v_v))    # claramente sobre el umbral

for intento in range(50):
    t_ext, trayectorias_v, tiempos_v = realizacion2(lambda_v, estado0_v, vecinos_v, pesos_v_v, s_v, T_v, guardar=True)
    if t_ext > 0.7*T_v:
        break

animar_epidemia(vecinos_v, pesos_v_v, trayectorias_v, tiempos_v, T_v, archivo_gif="epidemia_aleatorio.gif",
                etiquetas_peso={1.0: "familia (λ)", 0.5: "amigos (λ/2)", 0.333: "compañeros (λ/3)"})

# Video Simulación SCMA-26

In [ ]:
#######  ayuda de claude #######
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

# imageio-ffmpeg
try:
    import imageio_ffmpeg
    plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except ImportError:
    pass

# ---------------- utilidades para los videos ----------------
def guardar(anim, nombre, fps=6, pausa_final_ms=5000):
    """mp4 si hay ffmpeg; si no, gif que se reproduce una vez y se queda en el último cuadro."""
    from PIL import Image as PILImage, ImageSequence
    if animation.writers.is_available("ffmpeg"):
        anim.save(nombre + ".mp4", writer=FFMpegWriter(fps=fps, bitrate=2500), dpi=130)
        print("Guardado:", nombre + ".mp4")
    else:
        ruta = nombre + ".gif"
        anim.save(ruta, writer=PillowWriter(fps=fps), dpi=90)
        with PILImage.open(ruta) as im:
            frames = [f.copy() for f in ImageSequence.Iterator(im)]
        dur = [int(1000 / fps)] * len(frames)
        dur[-1] = pausa_final_ms
        frames[0].save(ruta, save_all=True, append_images=frames[1:], duration=dur)   # sin 'loop': una sola vez
        print("Guardado:", ruta, "(sin bucle)")

C_SANO, C_INF, C_PROF = "#55cc6f", "#940675", "#222222"
leyenda = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_SANO, markeredgecolor="k", markersize=9, label="Sano"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_INF, markeredgecolor="k", markersize=9, label="Infectado"),
]

In [ ]:
#######  ayuda de claude ( excepto realizacion2 ) #######

# =====================================================
# VIDEO 2: profesor + 59 alumnos, 15 semanas
# =====================================================

# ---------------- parámetros ----------------

SEMILLA     =1111

N_ALUM      = 59
C_PROM      = 5          # grado medio entre alumnos 
LAMBDA      = 2.25       # contagio por arista, igual para todos 
DT_DIA      = 0.08       # unidades de tiempo 
SEMANAS     = 15
DIAS = 14 * 7 + 4              
DIAS_CLASE  = (1, 3)                 
NOMBRES_DIA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
PROF        = 0                      
NOMBRE_PROF = "Avelio"
TITULO = "Sicosis colectiva por la moneda de Avelio (SCMA-26)"
FPS2        = 5
PAUSA_FINAL_S = 40                  



NOMBRES = [   
    "Alex Kri M.", "Alonso Contreras C.", "Aurora Llanos B.", "Benjamín Beissinger S.",
    "Benjamín Bravo N.", "Benjamín Gallardo A.", "Benjamín Ibarra Alemany", "Benjamin Muñoz",
    "Benjamin Quitral F.", "Benjamín Salazar H.", "Bianca S. Zamora Araya", "Branco Álvarez V.",
    "Bruno Guevara", "Dante Hormazabal Y.", "Diego Alonso Carter C.", "Diego Palacios B.",
    "Eloi Álvarez", "Emilio Lagos R.", "Esteban I. Ibarra Michea", "Facundo Solari",
    "Felipe Cornejo", "Felipe Silva", "Fran Loyola Reyes", "Gabriel Conejeros B.",
    "Ignacio Gómez G.", "Ilia Barrientos Opazo", "Isaac Lienqueo", "Javiera Robles P.",
    "Javier Toro B.", "Josefina Gaete P.", "Juan Sanhueza F.", "Luciano Avegno Cepeda",
    "Luciano Jerez A.", "M. Florencia Núñez", "Marco Ipinza Céspedes", "María Rubio",
    "Martina Pizarro D.", "Martín Maturana A.", "Martín Orellana Jara", "Martín Pérez F.",
    "Matías Salinas A.", "Matilda Cañas A.", "Maximiliano Lizana G.", "Nabil Orellana G.",
    "Nicolás Villarroel H.", "Osvaldo Cartagena", "Pablo Castillo", "Pascal Soto J.",
    "Rayen Lobos T.", "Roberto Salvo Cuitiño", "Rodrigo Mendoza P.", "Santiago Escobar U.",
    "Sara Quezada", "Sebastián Wulf G.", "Tomás Padilla G.", "Valentina Manfredi R.",
    "Xavier Sepúlveda C.", "Claudio José López Fernández", "Fraick Reyes",
]


# ---------------- grafo a mano ----------------
np.random.seed(SEMILLA)
n = N_ALUM + 1
A = np.zeros((n, n), dtype=bool)
alum = np.triu(np.random.rand(N_ALUM, N_ALUM) < C_PROM / N_ALUM, k=1)   # aristas al azar entre alumnos
A[1:, 1:] = alum | alum.T
A[0, 1:] = True                                                         # el profesor se conecta con todos
A[1:, 0] = True

A_libre = A.copy()                                                      # día sin clase: sin aristas del profesor
A_libre[0, :] = False
A_libre[:, 0] = False

def armar(A_):
    vec = [np.flatnonzero(A_[x]) for x in range(n)]
    pes = [np.ones(len(v)) for v in vec]                                # todas las aristas pesan 1
    s = np.array([len(v) for v in vec], dtype=float)
    return vec, pes, s

vecinos_clase, pesos_clase, s_clase = armar(A)
vecinos_libre, pesos_libre, s_libre = armar(A_libre)

# realizacion2 con 2 cambios 
def realizacion2_inmune(lambda_, estado0, vecinos, pesos_v, s, T, guardar=False, inmune=-1):
    estado = estado0.copy()
    t = 0.0
    if guardar:
        trayectorias = [estado.copy()]
        tiempos = [0.0]
    else:
        trayectorias, tiempos = None, None

    while estado.sum() > 0:
        infectados = np.flatnonzero(estado)
        curables = infectados[infectados != inmune]      # cambio 1: el inmune nunca se cura
        A_t = len(curables)
        sx = s[infectados]
        S = sx.sum()

        R = A_t + lambda_*S
        if R == 0:                                       # cambio 2: hoy no puede pasar nada
            break
        U1 = np.random.uniform(0, 1)
        delta_t = -np.log(U1)/R
        if t + delta_t > T:
            break

        t = t + delta_t
        U2 = np.random.uniform(0, 1)
        if U2 < A_t/R:
            x = curables[np.random.randint(A_t)]         # se cura un infectado 
            estado[x] = 0
        else:
            x = infectados[np.random.choice(len(infectados), p=sx/S)]
            v = np.random.choice(vecinos[x], p=pesos_v[x]/s[x])
            if estado[v] == 0:
                estado[v] = 1
        if guardar:
            trayectorias.append(estado.copy())
            tiempos.append(t)
    return estado, trayectorias, tiempos                 # devuelve el estado final




# ---------------- simulación día por día ----------------
estado = np.zeros(n, dtype=np.int8)
estado[PROF] = 1                                         # solo el profesor parte infectado
hist = [estado.copy()]                                   # hist[d+1] = estado al final del día d
for d in range(DIAS):
    clase = (d % 7) in DIAS_CLASE
    vec, pes, s = (vecinos_clase, pesos_clase, s_clase) if clase else (vecinos_libre, pesos_libre, s_libre)
    estado, _, _ = realizacion2_inmune(LAMBDA, estado, vec, pes, s, DT_DIA, inmune=PROF)
    hist.append(estado.copy())

est_final = hist[-1]          # estado al final del jueves de la semana 15
sanos_idx = [a for a in range(1, n) if est_final[a] == 0]



# ---------------- video ----------------
pos = {PROF: (0.0, 0.0)}
for a in range(1, n):
    ang = np.pi / 2 - 2 * np.pi * (a - 1) / N_ALUM
    pos[a] = (np.cos(ang), np.sin(ang))
xy = np.array([pos[v] for v in range(n)])

seg_al = [(pos[x], pos[v]) for x in range(1, n) for v in vecinos_clase[x] if v > x]
seg_pr = [(pos[PROF], pos[v]) for v in vecinos_clase[PROF]]

fig2, ax2 = plt.subplots(figsize=(13, 9))
ax2.set_aspect("equal")
fig2.subplots_adjust(left=0.01, right=0.66, top=0.86, bottom=0.02)
fig2.suptitle(TITULO, x=0.34, fontsize=15, fontweight="bold", y=0.975)

# panel con los nombres de los sobrevivientes (solo se ve en el estado final)
txt_sanos = f"Sobrevivientes ({len(sanos_idx)} de {N_ALUM})\n\n" + "\n".join(NOMBRES[a - 1] for a in sanos_idx)
fs = 11 if len(sanos_idx) <= 25 else 7
panel = fig2.text(0.69, 0.5, txt_sanos, va="center", ha="left", fontsize=fs,
                  family="monospace", visible=False)

ax2.set_axis_off(); ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2)
ax2.add_collection(LineCollection(seg_al, colors="gray", alpha=0.35, linewidths=1.0))
lc_prof = LineCollection(seg_pr, colors="orange", alpha=0.35, linewidths=0.7, linestyles="dashed")
ax2.add_collection(lc_prof)
sc2 = ax2.scatter(xy[:, 0], xy[:, 1], s=230, c=[C_SANO] * n, edgecolors="k", zorder=3)
for v in range(n):
    ax2.text(pos[v][0], pos[v][1], "P" if v == PROF else str(v), ha="center", va="center",
             fontsize=6.5, color="white", fontweight="bold", zorder=4)
titulo2 = ax2.set_title("", fontsize=12)
ax2.legend(handles=leyenda + [Line2D([0], [0], marker="o", color="w", markerfacecolor=C_PROF,
           markeredgecolor="k", markersize=9, label=f"Profesor {NOMBRE_PROF} (P): nunca se cura")],
           loc="lower right")

def update2(f):
    panel.set_visible(f == len(hist) - 1)
    est = hist[f]
    cols = [C_INF if e == 1 else C_SANO for e in est]
    cols[PROF] = C_PROF
    sc2.set_facecolor(cols)
    inf_al = int(est[1:].sum())
    if f == 0:
        lc_prof.set_visible(False)
        txt = f"Inicio (lunes, semana 1): solo el profesor {NOMBRE_PROF} está infectado"
    else:
        d = f - 1
        semana, dsem = d // 7 + 1, d % 7
        clase = dsem in DIAS_CLASE
        lc_prof.set_visible(clase)
        txt = (f"Semana {semana} – {NOMBRES_DIA[dsem]}" + ("  (HAY CLASE)" if clase else "") +
               f"\nAlumnos infectados: {inf_al}   sanos: {N_ALUM - inf_al}")
    titulo2.set_text(txt)
    return sc2,

frames2 = list(range(len(hist))) + [len(hist) - 1] * int(PAUSA_FINAL_S * FPS2)  
anim2 = FuncAnimation(fig2, update2, frames=frames2, interval=1000 / FPS2, blit=False)

update2(len(hist) - 1)                                                            
fig2.savefig("distribucion_final_profesor.png", dpi=150, bbox_inches="tight")

guardar(anim2, "video_profesor_15_semanas", fps=FPS2, pausa_final_ms=int(PAUSA_FINAL_S * 1000))
plt.close(fig2)

# ---------------- sobrevivientes ----------------
for a in sanos_idx:
    print(NOMBRES[a - 1])